# 02 – RL Agent Training (MaskablePPO + CNN)

Pełne podejście: `PacmanGridEnv` (obserwacja 2D `6×31×28` + maska legalnych akcji + reward shaping z PBRS),
algorytm **MaskablePPO** z małą siecią CNN, równoległe środowiska (`SubprocVecEnv`).

Naciśnij **⏹ Stop Training**, aby zatrzymać; checkpoint jest zapisywany co `CHECKPOINT_EVERY` kroków.

In [2]:
# Install sb3-contrib into THIS kernel (idempotent; %pip ensures correct interpreter).
import importlib, subprocess, sys
if importlib.util.find_spec("sb3_contrib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sb3-contrib"])
    print("sb3-contrib installed — restart NOT required, just re-run the next cell.")
else:
    print("sb3-contrib already available.")

sb3-contrib installed — restart NOT required, just re-run the next cell.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys, threading, time, os
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym

import ipywidgets as widgets
from IPython.display import display

from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecMonitor
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

from src.environment.pacman_env import PacmanGridEnv
from src.utils.mlflow_logger import MLflowLogger

In [4]:
# ── Configuration ────────────────────────────────────────────────────────────
N_ENVS           = 8           # parallel environments
N_STEPS          = 256         # rollout length per env
TOTAL_TIMESTEPS  = 5_000_000   # upper bound (Stop button can interrupt earlier)
CHECKPOINT_PATH  = os.path.join('..', 'models', 'ppo_pacman')
CHECKPOINT_EVERY = 100_000
LOG_EVERY        = 10_000

# ── Action-mask wrapper (MaskablePPO requires this on each env) ──────────────
def mask_fn(env):
    return env.action_masks()

def make_env(seed: int):
    def _f():
        env = PacmanGridEnv(
            seed=seed,
            max_steps=2000,
            step_penalty=-0.01,
            reward_scale_div=100.0,
            pbrs_coef=0.05,
        )
        env = ActionMasker(env, mask_fn)
        return env
    return _f

# Build vectorised env
env_fns = [make_env(seed=i) for i in range(N_ENVS)]
vec_env = SubprocVecEnv(env_fns) if N_ENVS > 1 else DummyVecEnv(env_fns)
vec_env = VecMonitor(vec_env)

# ── Custom small CNN suited for 6×31×28 input ────────────────────────────────
class PacmanCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        c, h, w = observation_space.shape
        self.cnn = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            n_flat = self.cnn(torch.zeros(1, c, h, w)).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flat, features_dim), nn.ReLU())

    def forward(self, x):
        return self.linear(self.cnn(x))

policy_kwargs = dict(
    features_extractor_class=PacmanCNN,
    features_extractor_kwargs=dict(features_dim=256),
    net_arch=dict(pi=[128, 128], vf=[128, 128]),
)

model = MaskablePPO(
    "CnnPolicy",
    vec_env,
    learning_rate=2.5e-4,
    n_steps=N_STEPS,
    batch_size=512,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.1,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    verbose=0,
    device="auto",
)

print(f"Model ready. Obs shape: {vec_env.observation_space.shape}, n_envs={N_ENVS}")
print(f"Device: {model.device}")

Model ready. Obs shape: (6, 31, 28), n_envs=8
Device: cuda


In [5]:
stop_event = threading.Event()

# ── UI ────────────────────────────────────────────────────────────────────────
stop_btn = widgets.Button(
    description='⏹  Stop Training',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='40px'),
)
status_label = widgets.Label(value='Status: not started')
log_output   = widgets.Output(layout=widgets.Layout(
    height='320px', overflow_y='auto', border='1px solid #ccc', padding='6px'
))

def on_stop_clicked(_):
    stop_event.set()
    stop_btn.description = '⏹  Stopping…'
    stop_btn.disabled = True
    status_label.value = 'Status: stopping after current rollout…'

stop_btn.on_click(on_stop_clicked)
display(widgets.HBox([stop_btn, status_label]), log_output)

# ── Callback ──────────────────────────────────────────────────────────────────
class TrainingCallback(BaseCallback):
    def __init__(self, stop_event, log_out, checkpoint_path,
                 checkpoint_every, log_every, mlflow_logger):
        super().__init__()
        self.stop_event       = stop_event
        self.log_out          = log_out
        self.checkpoint_path  = checkpoint_path
        self.checkpoint_every = checkpoint_every
        self.log_every        = log_every
        self.mlflow_logger    = mlflow_logger
        self._next_log        = log_every
        self._next_ckpt       = checkpoint_every
        self._ep_rewards      = []
        self._ep_lengths      = []
        self._t0              = time.time()

    def _on_step(self) -> bool:
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self._ep_rewards.append(info['episode']['r'])
                self._ep_lengths.append(info['episode']['l'])

        n = self.num_timesteps
        if n >= self._next_log:
            elapsed = time.time() - self._t0
            recent_r = self._ep_rewards[-50:]
            recent_l = self._ep_lengths[-50:]
            mean_r = float(np.mean(recent_r)) if recent_r else 0.0
            mean_l = float(np.mean(recent_l)) if recent_l else 0.0
            with self.log_out:
                print(f"[{n:>9,} steps | {elapsed/60:5.1f} min] "
                      f"mean_r(50)={mean_r:7.2f}  mean_len(50)={mean_l:6.0f}  "
                      f"episodes={len(self._ep_rewards)}")
            self.mlflow_logger.log_metrics(
                {'mean_reward_50ep': mean_r,
                 'mean_ep_length_50': mean_l,
                 'elapsed_min': elapsed / 60,
                 'episodes': len(self._ep_rewards)},
                step=n,
            )
            self._next_log = n + self.log_every

        if n >= self._next_ckpt:
            self.model.save(self.checkpoint_path)
            with self.log_out:
                print(f"  ✓ checkpoint saved → {self.checkpoint_path}.zip")
            self._next_ckpt = n + self.checkpoint_every

        return not self.stop_event.is_set()

# ── Training thread ───────────────────────────────────────────────────────────
def _run_training():
    status_label.value = 'Status: ▶ running…'
    with log_output:
        print(f"Training started — MaskablePPO + CNN, {N_ENVS} parallel envs.\n")

    with MLflowLogger(experiment_name='rl_training', run_name='ppo_cnn_masked') as logger:
        logger.log_params({
            'algorithm':         'MaskablePPO',
            'policy':            'CnnPolicy (custom 3xConv)',
            'n_envs':            N_ENVS,
            'n_steps':           N_STEPS,
            'batch_size':        512,
            'learning_rate':     2.5e-4,
            'gamma':             0.99,
            'gae_lambda':        0.95,
            'clip_range':        0.1,
            'ent_coef':          0.01,
            'env':               'PacmanGridEnv',
            'obs_shape':         str(vec_env.observation_space.shape),
            'reward_scale_div':  100.0,
            'pbrs_coef':         0.05,
            'step_penalty':      -0.01,
            'max_steps':         2000,
        })

        callback = TrainingCallback(
            stop_event, log_output, CHECKPOINT_PATH,
            CHECKPOINT_EVERY, LOG_EVERY, logger,
        )

        try:
            model.learn(
                total_timesteps=TOTAL_TIMESTEPS,
                callback=callback,
                reset_num_timesteps=True,
                progress_bar=False,
            )
        except Exception as exc:
            with log_output:
                print(f"[ERROR] {exc}")
            raise
        finally:
            model.save(CHECKPOINT_PATH)
            logger.log_metrics({'total_timesteps': model.num_timesteps})
            with log_output:
                print(f"\nFinal model saved → {CHECKPOINT_PATH}.zip")
            status_label.value = f'Status: stopped at {model.num_timesteps:,} steps'
            stop_btn.description = '⏹  Stopped'

train_thread = threading.Thread(target=_run_training, daemon=True)
train_thread.start()

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

In [4]:
# ── Optional: manually save or check progress at any time ────────────────────
model.save(CHECKPOINT_PATH)
print(f"Saved {model.num_timesteps:,} timesteps → {CHECKPOINT_PATH}.zip")
print(f"Thread alive: {train_thread.is_alive()}")

Saved 0 timesteps → ../models/dqn_pacman.zip
Thread alive: True
